# 2 - Tagging e Extração Estruturada com LangChain

**Tagging** é o processo de classificar ou rotular um texto automaticamente.
Em vez de o LLM retornar texto livre, forçamos ele a retornar dados **estruturados** (JSON)
que seguem um schema pré-definido.

Exemplos de uso:
- Detectar sentimento e idioma de um texto
- Classificar tickets de suporte por categoria
- Extrair entidades (nomes, datas, valores) de documentos

### O que mudou na API moderna

| Antigo (deprecado) | Moderno |
|--------------------|--------|
| `from langchain.pydantic_v1 import BaseModel` | `from pydantic import BaseModel` |
| `convert_to_openai_function(Modelo)` | não necessário |
| `chat.bind(functions=[...], function_call={...})` | `chat.with_structured_output(Modelo)` |
| `JsonOutputFunctionsParser()` | não necessário |

> **`with_structured_output()`** é o método padrão moderno. Ele cuida internamente de converter
> o schema Pydantic para o formato da API e de parsear a resposta — sem nenhum boilerplate extra.

### Pacotes necessários

```bash
pip install langchain-openai pydantic python-dotenv
```

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

chat = ChatOpenAI(model="gpt-4o-mini")

## 1. Definindo o schema com Pydantic

O schema é a "forma" que queremos que a resposta tenha.
Definimos uma classe Pydantic com os campos desejados e suas descrições.

As **descrições dos campos** (`Field(description=...)`) são fundamentais — elas são enviadas
ao modelo como instruções sobre o que cada campo deve conter.

> **Pydantic v2**: use `from pydantic import BaseModel, Field` diretamente.
> O antigo `from langchain.pydantic_v1 import BaseModel` foi removido.

In [2]:
from pydantic import BaseModel, Field

class Sentiment(BaseModel):
    """Define o sentimento e o idioma da mensagem enviada."""
    sentimento: str = Field(
        description="Sentimento do texto. Deve ser 'pos', 'neg' ou 'nt' para não definido"
    )
    lingua: str = Field(
        description="Língua em que o texto foi escrito (ex: 'pt', 'en', 'es')"
    )

## 2. Criando a chain com `with_structured_output()`

O método `.with_structured_output(Schema)` transforma o modelo em um componente que:
1. Converte o schema Pydantic para o formato de tool/function da API
2. Força o modelo a preencher exatamente aquele schema
3. Parseia a resposta e retorna uma **instância do objeto Pydantic** diretamente

```
prompt | chat.with_structured_output(Sentiment)
                        │
                        └── retorna Sentiment(sentimento='pos', lingua='pt')
                            em vez de AIMessage
```

> **Sem `StrOutputParser()`**: como a saída já é um objeto Pydantic (não um `AIMessage`),
> não precisamos de parser adicional.

In [3]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "Analise o texto e categorize-o conforme as instruções"),
    ("human", "{input}")
])

# .with_structured_output() substitui o antigo .bind(functions=[...], function_call={...})
# Retorna uma instância do modelo Pydantic, não um AIMessage
chain = prompt | chat.with_structured_output(Sentiment)

In [5]:
# Testando com texto em português (sentimento positivo)
texto = "Eu gosto muito de pizza nordestina"
resposta = chain.invoke({"input": texto})

print(f"Texto: '{texto}'")
print(f"Sentimento: {resposta.sentimento}")
print(f"Língua: {resposta.lingua}")
print(f"\nObjeto completo: {resposta}")

Texto: 'Eu gosto muito de pizza nordestina'
Sentimento: pos
Língua: pt

Objeto completo: sentimento='pos' lingua='pt'


In [6]:
# Testando com texto em inglês (sentimento neutro)
texto_en = "May the Force be with you"
resposta_en = chain.invoke({"input": texto_en})

print(f"Texto: '{texto_en}'")
print(f"Sentimento: {resposta_en.sentimento}")
print(f"Língua: {resposta_en.lingua}")

Texto: 'May the Force be with you'
Sentimento: nt
Língua: en


## 3. Usando Enum para valores controlados

Quando queremos restringir o modelo a um conjunto fixo de valores, usamos `Enum`.
Isso garante que a resposta seja sempre um dos valores permitidos — nada fora da lista.

É especialmente útil para **categorização** e **roteamento**, onde os valores possíveis
são conhecidos de antemão.

> **Por que Enum?** Sem ele, o modelo poderia retornar strings variadas como
> `"hardware"`, `"Hardware"`, `"problema de hardware"` etc. Com Enum, só aceita
> exatamente os valores definidos.

In [4]:
from enum import Enum
from pydantic import BaseModel, Field

class SetorEnum(str, Enum):
    suporte_hardware = "suporte_hardware"
    suporte_software = "suporte_software"
    suporte_conta    = "suporte_conta"
    outros           = "outros"

class DirecionaSuporte(BaseModel):
    """Direciona a solicitação de suporte para o setor responsável."""
    setor: SetorEnum = Field(
        description="Setor responsável por atender a solicitação"
    )

print("Schema definido. Valores possíveis para 'setor':")
for valor in SetorEnum:
    print(f"  - {valor.value}")

Schema definido. Valores possíveis para 'setor':
  - suporte_hardware
  - suporte_software
  - suporte_conta
  - outros


## 4. Chain de categorização com instruções detalhadas

Para casos de categorização, é importante dar **exemplos e critérios claros** no prompt de sistema.
O modelo usa essas instruções para decidir qual categoria aplicar em casos ambíguos.

Aqui substituímos o antigo `JsonOutputFunctionsParser()` por nada — o `with_structured_output()`
já retorna o objeto Pydantic diretamente.

In [7]:
system_message = """Pense com cuidado ao categorizar o texto conforme as instruções.
Questões relacionadas a problemas de hardware, como falha no computador, laptop ou 
outros dispositivos físicos devem ser direcionadas para 'suporte_hardware'.
Questões relacionadas a problemas com software, como instalação, erros ao abrir programas, 
etc., devem ser direcionadas para 'suporte_software'.
Problemas relacionados ao acesso de conta, como recuperação de senha, problemas de login, 
devem ser direcionados para 'suporte_conta'.
Mensagens que não se encaixam nessas categorias devem ser direcionadas para 'outros'.
"""

prompt_suporte = ChatPromptTemplate.from_messages([
    ("system", system_message),
    ("human", "{input}")
])

# Substitui o antigo: chat.bind(functions=[...]) | JsonOutputFunctionsParser()
chain_suporte = prompt_suporte | chat.with_structured_output(DirecionaSuporte)

In [8]:
# Lista de solicitações de suporte para testar
solicitacoes = [
    "Meu computador está travando toda vez que tento abrir um programa. O que devo fazer?",
    "Não consigo acessar minha conta. A senha não está funcionando!",
    "Estou tendo dificuldades para instalar o software de design gráfico. Pode me ajudar?",
    "Gostaria de saber como atualizar meu endereço de entrega na plataforma.",
    "O meu laptop não está ligando. Acho que é um problema no cabo de energia.",
    "Não consigo ver meus arquivos no aplicativo. Existe alguma solução?"
]

In [9]:
# Testando casos individuais
for i in [0, 3, 4]:  # software, conta, hardware
    solicitacao = solicitacoes[i]
    resposta = chain_suporte.invoke({"input": solicitacao})
    print(f"Solicitação: {solicitacao}")
    print(f"Setor: {resposta.setor.value}")
    print()

Solicitação: Meu computador está travando toda vez que tento abrir um programa. O que devo fazer?
Setor: suporte_software

Solicitação: Gostaria de saber como atualizar meu endereço de entrega na plataforma.
Setor: outros

Solicitação: O meu laptop não está ligando. Acho que é um problema no cabo de energia.
Setor: suporte_hardware



## 5. Processando múltiplos textos em lote

O LCEL oferece `.batch()` para processar uma lista de inputs de uma vez,
de forma mais eficiente do que um loop com `.invoke()`.

| Método | Uso | Comportamento |
|--------|-----|---------------|
| `.invoke(input)` | Um item | Síncrono, retorna um resultado |
| `.batch(lista)` | Múltiplos itens | Paralelo, retorna lista de resultados |
| `.stream(input)` | Um item | Streaming token a token |

In [10]:
# .batch() processa todas as solicitações de forma eficiente
inputs = [{"input": s} for s in solicitacoes]
resultados = chain_suporte.batch(inputs)

print("Resultado do direcionamento automático:\n")
for solicitacao, resultado in zip(solicitacoes, resultados):
    # Truncando a solicitação para exibição
    trecho = solicitacao[:60] + "..."
    print(f"[{resultado.setor.value:>20}]  {trecho}")

Resultado do direcionamento automático:

[    suporte_software]  Meu computador está travando toda vez que tento abrir um pro...
[       suporte_conta]  Não consigo acessar minha conta. A senha não está funcionand...
[    suporte_software]  Estou tendo dificuldades para instalar o software de design ...
[              outros]  Gostaria de saber como atualizar meu endereço de entrega na ...
[    suporte_hardware]  O meu laptop não está ligando. Acho que é um problema no cab...
[    suporte_software]  Não consigo ver meus arquivos no aplicativo. Existe alguma s...


## Resumo

| Conceito | Descrição |
|----------|-----------|
| `BaseModel` (Pydantic v2) | Define o schema da saída estruturada |
| `Field(description=...)` | Instrui o modelo sobre o que cada campo deve conter |
| `Enum` | Restringe os valores possíveis de um campo |
| `with_structured_output(Schema)` | Força o modelo a retornar o schema definido |
| `.batch(lista)` | Processa múltiplos inputs eficientemente |

**Padrão moderno completo:**

```python
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

class MeuSchema(BaseModel):
    """Descrição do que o schema representa."""
    campo: str = Field(description="O que este campo deve conter")

chain = prompt | chat.with_structured_output(MeuSchema)
resultado = chain.invoke({"input": "texto a analisar"})
# resultado é uma instância de MeuSchema
print(resultado.campo)
```

> **Próximo passo**: usar `with_structured_output()` para **extração de informações**
> — identificar e extrair entidades específicas de documentos longos (nomes, datas, valores, etc.).